# Poor Window Mechanism Router v1

Notebook-owned diagnostic to decide the next optimization mechanism after several execution overlays reduced some risk but still missed the `+30pp excess / <15% MaxDD` target.

Governance:
- Baselines: production paper-model poor-window diagnostic and `recovery_reentry_regime_drawdown_guard_v1`
- Direct comparisons: anchor-confirmed reentry, equity drawdown guard, CPPI floor, 2021 trend-break cap, SOXL/SOXX chop guard
- Role: `diagnose`
- Changed layer: `execution` / `model_selection` decision routing, no new training and no shared-code change
- Target window: strict OOS poor windows plus saved walk-forward sidecars where available
- Accepted trade-off: this run may not improve returns directly; it must reduce overfit risk by ruling out duplicate cap tweaks and identifying the next mechanism
- Comparison row key: `poor_window_mechanism_router_v1`

External basis:
- Probability of Backtest Overfitting: repeated trials need an explicit selection-bias guard, not another threshold tweak. https://doi.org/10.21314/JCF.2016.322
- Deep Momentum Networks: separate trend estimation from position sizing and downside-aware scaling. https://arxiv.org/abs/1904.04912
- Volatility-Managed Portfolios: volatility scaling can help, but must be conditioned to avoid underparticipation. https://www.nber.org/papers/w22208
- Slow Momentum with Fast Reversion: fast adverse moves should be handled separately from slower trend exposure. https://arxiv.org/abs/2105.13727
- Qlib: keep diagnostics and research-to-runtime artifacts in the qlib harness. https://github.com/microsoft/qlib

Historical success/failure reflection:
- Closest success: anchor-confirmed reentry removed `wf_2022` false positives while preserving a meaningful 2026 rebound improvement.
- Closest failure: equity/CPPI/trend-break caps improved or preserved some strict-OOS metrics but left MaxDD near 30%+ or introduced walk-forward regression.
- This run is not a duplicate because it does not test another threshold; it aggregates cross-card poor-window evidence to choose the next mechanism.

Hypothesis:
Persistent bad windows are high-volatility rebound underexposure clusters, not a single drawdown-cap problem. If true, the next useful experiment is a state router that separates high-volatility positive rebound participation from high-volatility downtrend de-risking, while preserving `wf_2022` defenses.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')

CARD = 'poor_window_mechanism_router_v1'
CARD_DIR = ROOT / 'runtime' / 'execution' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_EXCESS = 30.0
TARGET_MAX_DD = 15.0

SOURCES = [
    {
        'label': 'paper_model',
        'card': 'oos_poor_window_feature_diagnostics_v1',
        'direction': 'production',
        'poor_windows': ROOT / 'runtime/production/card/oos_poor_window_feature_diagnostics_v1/poor_windows.csv',
        'features': ROOT / 'runtime/production/card/oos_poor_window_feature_diagnostics_v1/feature_differences.csv',
        'selection': None,
        'role': 'runtime_baseline',
    },
    {
        'label': 'anchor_confirm',
        'card': 'recovery_reentry_anchor_confirm_v1',
        'direction': 'execution',
        'poor_windows': ROOT / 'runtime/execution/card/recovery_reentry_anchor_confirm_v1/oos_poor_window_diagnostics/poor_windows.csv',
        'features': ROOT / 'runtime/execution/card/recovery_reentry_anchor_confirm_v1/oos_poor_window_diagnostics/feature_differences.csv',
        'selection': None,
        'role': 'rebound_reentry_control',
    },
    {
        'label': 'equity_guard',
        'card': 'recovery_reentry_equity_drawdown_guard_v1',
        'direction': 'execution',
        'poor_windows': ROOT / 'runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/oos_poor_window_diagnostics/poor_windows.csv',
        'features': ROOT / 'runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/oos_poor_window_diagnostics/feature_differences.csv',
        'selection': None,
        'role': 'path_level_dd_control',
    },
    {
        'label': 'regime_guard',
        'card': 'recovery_reentry_regime_drawdown_guard_v1',
        'direction': 'execution',
        'poor_windows': ROOT / 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/oos_poor_window_diagnostics/poor_windows.csv',
        'features': ROOT / 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/oos_poor_window_diagnostics/feature_differences.csv',
        'selection': ROOT / 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/selection.json',
        'role': 'regime_dd_control',
    },
    {
        'label': 'cppi_floor',
        'card': 'recovery_reentry_cppi_floor_v1',
        'direction': 'execution',
        'poor_windows': ROOT / 'runtime/execution/card/recovery_reentry_cppi_floor_v1/oos_poor_window_diagnostics/poor_windows.csv',
        'features': ROOT / 'runtime/execution/card/recovery_reentry_cppi_floor_v1/oos_poor_window_diagnostics/feature_differences.csv',
        'selection': ROOT / 'runtime/execution/card/recovery_reentry_cppi_floor_v1/selection.json',
        'role': 'floor_control',
    },
    {
        'label': 'trendbreak_cap',
        'card': 'recovery_reentry_2021_trendbreak_cap_v1',
        'direction': 'execution',
        'poor_windows': ROOT / 'runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/oos_poor_window_diagnostics/poor_windows.csv',
        'features': ROOT / 'runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/oos_poor_window_diagnostics/feature_differences.csv',
        'selection': ROOT / 'runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/selection.json',
        'role': 'trendbreak_cap',
    },
    {
        'label': 'soxx_chop',
        'card': 'soxl_soxx_efficiency_chop_guard_v1',
        'direction': 'execution',
        'poor_windows': ROOT / 'runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/oos_poor_window_diagnostics/poor_windows.csv',
        'features': ROOT / 'runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/oos_poor_window_diagnostics/feature_differences.csv',
        'selection': None,
        'role': 'relative_value_guard',
    },
]

for src in SOURCES:
    assert src['poor_windows'].exists(), src['poor_windows']
    assert src['features'].exists(), src['features']
print('sources', [src['label'] for src in SOURCES])


sources ['paper_model', 'anchor_confirm', 'equity_guard', 'regime_guard', 'cppi_floor', 'trendbreak_cap', 'soxx_chop']


In [2]:
def classify_window(start: str, end: str) -> str:
    s = pd.Timestamp(start)
    e = pd.Timestamp(end)
    if s >= pd.Timestamp('2026-01-01'):
        return '2026_high_vol_rebound_underexposure'
    if s >= pd.Timestamp('2025-04-01') and e <= pd.Timestamp('2025-08-31'):
        return '2025_high_vol_rebound_underexposure'
    if s.year == 2024 or e.year == 2024:
        return '2024_walkforward_recovery'
    if s.year == 2021 or e.year == 2021:
        return '2021_strict_oos_drawdown'
    if s.year == 2022 or e.year == 2022:
        return '2022_bear_defense'
    return 'other'

poor_frames = []
feature_frames = []
selection_rows = []
for src in SOURCES:
    poor = pd.read_csv(src['poor_windows'])
    poor['source'] = src['label']
    poor['card'] = src['card']
    poor['role'] = src['role']
    poor['cluster'] = [classify_window(s, e) for s, e in zip(poor['start'], poor['end'])]
    poor['excess_gap_to_target'] = TARGET_EXCESS - poor['excess_return']
    poor['drawdown_gap_to_target'] = poor['max_dd'] - TARGET_MAX_DD
    poor['underexposure_gap'] = poor['buyhold_return'] - poor['strategy_return']
    poor['target_excess_fail'] = poor['excess_return'] < TARGET_EXCESS
    poor['target_drawdown_fail'] = poor['max_dd'] > TARGET_MAX_DD
    poor_frames.append(poor)

    features = pd.read_csv(src['features'])
    features['source'] = src['label']
    features['card'] = src['card']
    features['role'] = src['role']
    feature_frames.append(features)

    if src['selection'] and src['selection'].exists():
        payload = json.loads(src['selection'].read_text())
        selected = payload.get('selected_strict') or payload.get('selected') or {}
        acceptance = payload.get('acceptance') or {}
        selection_rows.append({
            'source': src['label'],
            'card': src['card'],
            'decision': payload.get('decision'),
            'status': payload.get('status'),
            'strict_return': selected.get('return'),
            'strict_relative_return': selected.get('relative_return'),
            'strict_max_dd': selected.get('max_dd'),
            'strict_avg_pos': selected.get('avg_pos'),
            'strict_relative_pass': acceptance.get('strict_relative_pass'),
            'strict_max_dd_pass': acceptance.get('strict_max_dd_pass'),
            'wf_2022_ok': acceptance.get('wf_2022_ok'),
        })

poor_all = pd.concat(poor_frames, ignore_index=True)
features_all = pd.concat(feature_frames, ignore_index=True)
selection_summary = pd.DataFrame(selection_rows)

poor_inventory_path = CARD_DIR / 'poor_window_inventory.csv'
poor_all.to_csv(poor_inventory_path, index=False)
feature_inventory_path = CARD_DIR / 'feature_difference_inventory.csv'
features_all.to_csv(feature_inventory_path, index=False)
selection_summary_path = CARD_DIR / 'source_selection_summary.csv'
selection_summary.to_csv(selection_summary_path, index=False)

print('poor windows', poor_all.shape, poor_inventory_path.relative_to(ROOT))
print('feature differences', features_all.shape, feature_inventory_path.relative_to(ROOT))
display(poor_all.sort_values('bad_score', ascending=False).head(12))


poor windows (84, 18) runtime/execution/card/poor_window_mechanism_router_v1/poor_window_inventory.csv
feature differences (210, 12) runtime/execution/card/poor_window_mechanism_router_v1/feature_difference_inventory.csv


,start,end,rows,strategy_return,buyhold_return,excess_return,max_dd,avg_pos,bad_score,source,card,role,cluster,excess_gap_to_target,drawdown_gap_to_target,underexposure_gap,target_excess_fail,target_drawdown_fail
0,2026-02-10,2026-05-11,63,1.742286,200.631509,-198.889223,6.421473,0.139833,205.310696,paper_model,oos_poor_window_feature_diagnostics_v1,runtime_baseline,2026_high_vol_rebound_underexposure,228.889223,-8.578527,198.889223,True,False
1,2026-02-05,2026-05-06,63,17.689445,211.455411,-193.765966,6.421473,0.179353,200.187438,paper_model,oos_poor_window_feature_diagnostics_v1,runtime_baseline,2026_high_vol_rebound_underexposure,223.765966,-8.578527,193.765966,True,False
2,2026-02-12,2026-05-13,63,1.985132,191.980998,-189.995867,6.421473,0.113486,196.417340,paper_model,oos_poor_window_feature_diagnostics_v1,runtime_baseline,2026_high_vol_rebound_underexposure,219.995867,-8.578527,189.995867,True,False
48,2025-04-08,2025-07-09,63,38.928007,223.030298,-184.102291,11.499213,0.384934,195.601504,cppi_floor,recovery_reentry_cppi_floor_v1,floor_control,2025_high_vol_rebound_underexposure,214.102291,-3.500787,184.102291,True,False
3,2026-02-13,2026-05-14,63,0.024917,188.264457,-188.239541,6.421473,0.100313,194.661013,paper_model,oos_poor_window_feature_diagnostics_v1,runtime_baseline,2026_high_vol_rebound_underexposure,218.239541,-8.578527,188.239541,True,False
60,2026-02-05,2026-05-06,63,24.806006,211.455411,-186.649405,4.516156,0.146825,191.165560,trendbreak_cap,recovery_reentry_2021_trendbreak_cap_v1,trendbreak_cap,2026_high_vol_rebound_underexposure,216.649405,-10.483844,186.649405,True,False
72,2026-02-05,2026-05-06,63,27.993920,211.455411,-183.461491,6.421473,0.182540,189.882963,soxx_chop,soxl_soxx_efficiency_chop_guard_v1,relative_value_guard,2026_high_vol_rebound_underexposure,213.461491,-8.578527,183.461491,True,False
24,2026-02-05,2026-05-06,63,28.805017,211.455411,-182.650394,4.536823,0.179840,187.187217,equity_guard,recovery_reentry_equity_drawdown_guard_v1,path_level_dd_control,2026_high_vol_rebound_underexposure,212.650394,-10.463177,182.650394,True,False
49,2025-04-21,2025-07-21,63,42.040671,215.729046,-173.688375,11.499213,0.479312,185.187588,cppi_floor,recovery_reentry_cppi_floor_v1,floor_control,2025_high_vol_rebound_underexposure,203.688375,-3.500787,173.688375,True,False
73,2026-02-10,2026-05-11,63,23.317153,200.631509,-177.314356,6.421473,0.175397,183.735829,soxx_chop,soxl_soxx_efficiency_chop_guard_v1,relative_value_guard,2026_high_vol_rebound_underexposure,207.314356,-8.578527,177.314356,True,False


In [3]:
cluster_summary = (
    poor_all.groupby('cluster')
    .agg(
        rows=('start', 'size'),
        sources=('source', 'nunique'),
        median_bad_score=('bad_score', 'median'),
        max_bad_score=('bad_score', 'max'),
        median_excess=('excess_return', 'median'),
        worst_excess=('excess_return', 'min'),
        median_strategy_return=('strategy_return', 'median'),
        median_buyhold_return=('buyhold_return', 'median'),
        median_max_dd=('max_dd', 'median'),
        median_avg_pos=('avg_pos', 'median'),
        excess_fail_rate=('target_excess_fail', 'mean'),
        dd_fail_rate=('target_drawdown_fail', 'mean'),
    )
    .reset_index()
    .sort_values(['sources', 'median_bad_score'], ascending=[False, False])
)

# Best observed intervention within each repeated cluster, by lowest bad_score.
best_rows = []
for cluster, frame in poor_all.groupby('cluster'):
    idx = frame['bad_score'].idxmin()
    row = frame.loc[idx]
    best_rows.append({
        'cluster': cluster,
        'best_source': row['source'],
        'best_bad_score': row['bad_score'],
        'best_strategy_return': row['strategy_return'],
        'best_excess_return': row['excess_return'],
        'best_max_dd': row['max_dd'],
        'best_avg_pos': row['avg_pos'],
        'best_start': row['start'],
        'best_end': row['end'],
    })
best_by_cluster = pd.DataFrame(best_rows)
cluster_summary = cluster_summary.merge(best_by_cluster, on='cluster', how='left')
cluster_summary_path = CARD_DIR / 'cluster_summary.csv'
cluster_summary.to_csv(cluster_summary_path, index=False)

display(cluster_summary)
print(cluster_summary_path.relative_to(ROOT))


,cluster,rows,sources,median_bad_score,max_bad_score,median_excess,worst_excess,median_strategy_return,median_buyhold_return,median_max_dd,...,excess_fail_rate,dd_fail_rate,best_source,best_bad_score,best_strategy_return,best_excess_return,best_max_dd,best_avg_pos,best_start,best_end
0,2026_high_vol_rebound_underexposure,56,7,170.813824,205.310696,-165.554164,-198.889223,21.452003,182.301841,6.421473,...,1.0,0.000000,anchor_confirm,143.560341,17.997101,-137.138869,6.421473,0.181584,2026-02-17,2026-05-15
1,2025_high_vol_rebound_underexposure,27,7,161.753367,195.601504,-150.676327,-184.102291,57.330125,215.729046,11.499213,...,1.0,0.074074,soxx_chop,146.674433,57.330125,-135.175220,11.499213,0.515217,2025-04-16,2025-07-17
2,other,1,1,148.545011,148.545011,-131.375155,-131.375155,57.475545,188.850700,17.169856,...,1.0,1.000000,trendbreak_cap,148.545011,57.475545,-131.375155,17.169856,0.499194,2020-03-20,2020-06-18


runtime/execution/card/poor_window_mechanism_router_v1/cluster_summary.csv


In [4]:
def feature_group(feature: str) -> str:
    f = feature.lower()
    if 'vol' in f or 'kurtosis' in f or 'jump' in f or 'coef' in f:
        return 'volatility_path'
    if 'mom' in f or 'strength' in f or 'decay' in f or 'anchor' in f or 'trend' in f:
        return 'trend_rebound'
    if 'tracking_error' in f or f.startswith('te_') or '_te_' in f:
        return 'tracking_error'
    if 'vix' in f:
        return 'vix_regime'
    if 'downside' in f or 'drawdown' in f or 'drop' in f:
        return 'downside_shape'
    if 'volume' in f or 'flow' in f or 'liquidity' in f:
        return 'liquidity'
    return 'other'

features_all['feature_group'] = features_all['feature'].map(feature_group)
features_all['delta_sign'] = np.sign(features_all['delta_z']).astype(int)
# Focus on the features each card itself ranked as most different.
top_features = features_all.sort_values(['source', 'abs_delta_z'], ascending=[True, False]).groupby('source').head(8)
feature_consensus = (
    top_features.groupby(['feature', 'feature_group'])
    .agg(
        top_count=('source', 'nunique'),
        mean_delta_z=('delta_z', 'mean'),
        mean_abs_delta_z=('abs_delta_z', 'mean'),
        sign_sum=('delta_sign', 'sum'),
        sources=('source', lambda x: ','.join(sorted(set(x)))),
    )
    .reset_index()
)
feature_consensus['sign_consistency'] = feature_consensus['sign_sum'].abs() / feature_consensus['top_count']
feature_consensus = feature_consensus.sort_values(['top_count', 'mean_abs_delta_z'], ascending=[False, False])

feature_group_consensus = (
    top_features.groupby('feature_group')
    .agg(
        top_mentions=('feature', 'size'),
        source_count=('source', 'nunique'),
        mean_abs_delta_z=('abs_delta_z', 'mean'),
        features=('feature', lambda x: ','.join(sorted(set(x))))
    )
    .reset_index()
    .sort_values(['source_count', 'mean_abs_delta_z'], ascending=[False, False])
)

feature_consensus_path = CARD_DIR / 'feature_consensus.csv'
feature_group_path = CARD_DIR / 'feature_group_consensus.csv'
feature_consensus.to_csv(feature_consensus_path, index=False)
feature_group_consensus.to_csv(feature_group_path, index=False)

display(feature_consensus.head(20))
display(feature_group_consensus)
print(feature_consensus_path.relative_to(ROOT), feature_group_path.relative_to(ROOT))


,feature,feature_group,top_count,mean_delta_z,mean_abs_delta_z,sign_sum,sources,sign_consistency
14,vol_of_vol_21,volatility_path,7,1.409613,1.409613,7,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
8,realized_vol_63,volatility_path,7,1.302179,1.302179,7,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
3,jump_variation_5,volatility_path,6,1.241739,1.241739,6,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
5,realized_kurtosis_21,volatility_path,6,0.952535,0.952535,6,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
4,mom_21,trend_rebound,6,0.795475,0.795475,6,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
1,downside_ratio_21,downside_shape,6,-0.784950,0.784950,-6,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
2,expectancy_proxy_21,other,6,0.784266,0.784266,6,"anchor_confirm,cppi_floor,equity_guard,paper_m...",1.0
9,relative_strength_21,trend_rebound,5,0.802459,0.802459,5,"anchor_confirm,equity_guard,paper_model,regime...",1.0
0,decay_63,trend_rebound,1,-2.833425,2.833425,-1,trendbreak_cap,1.0
11,tracking_error_vol_21,volatility_path,1,1.662115,1.662115,1,trendbreak_cap,1.0


,feature_group,top_mentions,source_count,mean_abs_delta_z,features
5,volatility_path,30,7,1.266215,"jump_variation_5,realized_kurtosis_21,realized..."
3,trend_rebound,12,7,0.968214,"decay_63,mom_21,relative_strength_21"
0,downside_shape,6,6,0.784950,downside_ratio_21
1,other,6,6,0.784266,expectancy_proxy_21
2,tracking_error,1,1,1.445182,tracking_error_21
4,vix_regime,1,1,0.743305,vix_percentile_252


runtime/execution/card/poor_window_mechanism_router_v1/feature_consensus.csv runtime/execution/card/poor_window_mechanism_router_v1/feature_group_consensus.csv


In [5]:
# Measure whether the latest cap-style branches actually changed the repeated rebound clusters.
pivot = (
    poor_all[poor_all['cluster'].isin(['2026_high_vol_rebound_underexposure', '2025_high_vol_rebound_underexposure'])]
    .groupby(['cluster', 'source'])
    .agg(
        mean_bad_score=('bad_score', 'mean'),
        mean_excess=('excess_return', 'mean'),
        mean_strategy_return=('strategy_return', 'mean'),
        mean_buyhold_return=('buyhold_return', 'mean'),
        mean_max_dd=('max_dd', 'mean'),
        mean_avg_pos=('avg_pos', 'mean'),
        windows=('start', 'count'),
    )
    .reset_index()
)
base = pivot[pivot['source'] == 'paper_model'][['cluster', 'mean_bad_score', 'mean_excess', 'mean_strategy_return', 'mean_avg_pos']].rename(
    columns={
        'mean_bad_score': 'paper_bad_score',
        'mean_excess': 'paper_excess',
        'mean_strategy_return': 'paper_strategy_return',
        'mean_avg_pos': 'paper_avg_pos',
    }
)
pivot = pivot.merge(base, on='cluster', how='left')
pivot['bad_score_delta_vs_paper'] = pivot['mean_bad_score'] - pivot['paper_bad_score']
pivot['excess_delta_vs_paper'] = pivot['mean_excess'] - pivot['paper_excess']
pivot['strategy_delta_vs_paper'] = pivot['mean_strategy_return'] - pivot['paper_strategy_return']
pivot['avg_pos_delta_vs_paper'] = pivot['mean_avg_pos'] - pivot['paper_avg_pos']
pivot = pivot.sort_values(['cluster', 'bad_score_delta_vs_paper'])
rebound_comparison_path = CARD_DIR / 'rebound_cluster_comparison.csv'
pivot.to_csv(rebound_comparison_path, index=False)
display(pivot)
print(rebound_comparison_path.relative_to(ROOT))


,cluster,source,mean_bad_score,mean_excess,mean_strategy_return,mean_buyhold_return,mean_max_dd,mean_avg_pos,windows,paper_bad_score,paper_excess,paper_strategy_return,paper_avg_pos,bad_score_delta_vs_paper,excess_delta_vs_paper,strategy_delta_vs_paper,avg_pos_delta_vs_paper
0,2025_high_vol_rebound_underexposure,anchor_confirm,155.495683,-141.093861,78.285811,219.379672,14.401822,0.567223,2,155.495683,-141.093861,78.285811,0.567223,0.000000,0.000000,0.000000,0.000000
3,2025_high_vol_rebound_underexposure,paper_model,155.495683,-141.093861,78.285811,219.379672,14.401822,0.567223,2,155.495683,-141.093861,78.285811,0.567223,0.000000,0.000000,0.000000,0.000000
5,2025_high_vol_rebound_underexposure,soxx_chop,158.630368,-147.131156,59.951322,207.082478,11.499213,0.508650,4,155.495683,-141.093861,78.285811,0.567223,3.134685,-6.037295,-18.334489,-0.058573
4,2025_high_vol_rebound_underexposure,regime_guard,158.630368,-147.131156,59.951322,207.082478,11.499213,0.508650,4,155.495683,-141.093861,78.285811,0.567223,3.134685,-6.037295,-18.334489,-0.058573
6,2025_high_vol_rebound_underexposure,trendbreak_cap,162.615680,-151.116467,60.825055,211.941522,11.499213,0.506461,3,155.495683,-141.093861,78.285811,0.567223,7.119997,-10.022606,-17.460756,-0.060761
2,2025_high_vol_rebound_underexposure,equity_guard,163.592367,-155.775367,47.232661,203.008028,7.817001,0.454791,5,155.495683,-141.093861,78.285811,0.567223,8.096684,-14.681506,-31.053150,-0.112431
1,2025_high_vol_rebound_underexposure,cppi_floor,170.999309,-159.500097,38.965916,198.466013,11.499213,0.431118,7,155.495683,-141.093861,78.285811,0.567223,15.503626,-18.406236,-39.319895,-0.136105
7,2026_high_vol_rebound_underexposure,anchor_confirm,161.249140,-154.659994,23.070167,177.730161,6.589146,0.199629,10,181.060056,-174.638583,3.091578,0.117962,-19.810916,19.978589,19.978589,0.081667
11,2026_high_vol_rebound_underexposure,regime_guard,165.424740,-158.793676,24.787095,183.580771,6.631064,0.201283,8,181.060056,-174.638583,3.091578,0.117962,-15.635316,15.844907,21.695518,0.083321
9,2026_high_vol_rebound_underexposure,equity_guard,167.398025,-162.472200,24.751005,187.223205,4.925825,0.173064,7,181.060056,-174.638583,3.091578,0.117962,-13.662031,12.166383,21.659428,0.055102


runtime/execution/card/poor_window_mechanism_router_v1/rebound_cluster_comparison.csv


In [6]:
persistent_clusters = cluster_summary[(cluster_summary['sources'] >= 5) & (cluster_summary['excess_fail_rate'] >= 0.8)].copy()
volatility_consensus = feature_group_consensus[feature_group_consensus['feature_group'] == 'volatility_path']
trend_consensus = feature_group_consensus[feature_group_consensus['feature_group'] == 'trend_rebound']
cap_like_sources = {'equity_guard', 'regime_guard', 'cppi_floor', 'trendbreak_cap', 'soxx_chop'}
cap_like = pivot[pivot['source'].isin(cap_like_sources)]
cap_remaining_negative_excess = bool((cap_like['mean_excess'] < TARGET_EXCESS).all()) if not cap_like.empty else True
maxdd_pass_sources = selection_summary[selection_summary.get('strict_max_dd_pass', pd.Series(dtype=bool)) == True] if not selection_summary.empty else pd.DataFrame()

if not persistent_clusters.empty and cap_remaining_negative_excess:
    decision = 'diagnostic_next_step_rebound_router_no_promotion'
    next_mechanism = 'state-conditioned rebound participation router with separate high-vol positive-rebound and high-vol downtrend branches'
else:
    decision = 'diagnostic_inconclusive_need_more_oos_windows'
    next_mechanism = 'collect more OOS poor-window evidence before another mechanism test'

acceptance = {
    'repeated_clusters_detected': int(len(persistent_clusters)),
    'cap_like_sources_still_fail_rebound_excess': cap_remaining_negative_excess,
    'top_feature_group': feature_group_consensus.iloc[0]['feature_group'] if not feature_group_consensus.empty else None,
    'strict_maxdd_pass_sources': int(len(maxdd_pass_sources)),
}

readout_lines = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f'- Decision: `{decision}`',
    f'- Next mechanism: `{next_mechanism}`',
    f'- Repeated poor-window clusters: `{", ".join(persistent_clusters["cluster"].tolist()) if not persistent_clusters.empty else "none"}`',
    f'- Top feature group: `{acceptance["top_feature_group"]}`',
    f'- Cap-like sources still fail rebound excess: `{cap_remaining_negative_excess}`',
    f'- Strict MaxDD pass sources among structured selections: `{acceptance["strict_maxdd_pass_sources"]}`',
    '',
    '### Mechanism Notes',
    '',
    '- The persistent 2025/2026 clusters have large negative excess while drawdown is often already below the 15% target, so the remaining failure is underparticipation in violent rebounds, not only account drawdown.',
    '- Feature consensus is dominated by volatility-path and trend/rebound variables. `vol_of_vol_21`, `jump_variation_5`, `realized_vol_63`, and trend/rebound terms recur across cards.',
    '- Tightening execution caps alone is now a duplicated failure mode: it leaves rebound underexposure while risking `wf_2022` or trend-window regression.',
    '- The next experiment should route state, not tune another scalar cap: allow rebound participation only when high-volatility positive momentum and long-anchor recovery are both present; de-risk separately when high volatility is paired with trend breaks/downside momentum.',
    '',
    '### Artifacts',
    '',
    f'- `{poor_inventory_path.relative_to(ROOT)}`',
    f'- `{cluster_summary_path.relative_to(ROOT)}`',
    f'- `{feature_consensus_path.relative_to(ROOT)}`',
    f'- `{feature_group_path.relative_to(ROOT)}`',
    f'- `{rebound_comparison_path.relative_to(ROOT)}`',
]

selection_payload = {
    'card': CARD,
    'decision': decision,
    'status': 'diagnostic_no_runtime_promotion',
    'role': 'diagnose',
    'changed_layer': 'execution/model_selection',
    'target': {'excess_return_pp': TARGET_EXCESS, 'max_dd_pct': TARGET_MAX_DD},
    'acceptance': acceptance,
    'next_mechanism': next_mechanism,
    'persistent_clusters': persistent_clusters.to_dict(orient='records'),
    'top_feature_consensus': feature_consensus.head(12).to_dict(orient='records'),
    'feature_group_consensus': feature_group_consensus.to_dict(orient='records'),
    'rebound_cluster_comparison': pivot.to_dict(orient='records'),
    'artifacts': {
        'notebook': str((CARD_DIR / f'{CARD}.ipynb').relative_to(ROOT)),
        'poor_window_inventory': str(poor_inventory_path.relative_to(ROOT)),
        'feature_difference_inventory': str(feature_inventory_path.relative_to(ROOT)),
        'source_selection_summary': str(selection_summary_path.relative_to(ROOT)),
        'cluster_summary': str(cluster_summary_path.relative_to(ROOT)),
        'feature_consensus': str(feature_consensus_path.relative_to(ROOT)),
        'feature_group_consensus': str(feature_group_path.relative_to(ROOT)),
        'rebound_cluster_comparison': str(rebound_comparison_path.relative_to(ROOT)),
    },
    'external_basis': [
        {'title': 'Probability of Backtest Overfitting', 'url': 'https://doi.org/10.21314/JCF.2016.322'},
        {'title': 'Deep Momentum Networks', 'url': 'https://arxiv.org/abs/1904.04912'},
        {'title': 'Volatility-Managed Portfolios', 'url': 'https://www.nber.org/papers/w22208'},
        {'title': 'Slow Momentum with Fast Reversion', 'url': 'https://arxiv.org/abs/2105.13727'},
        {'title': 'Qlib', 'url': 'https://github.com/microsoft/qlib'},
    ],
}
selection_path = CARD_DIR / 'selection.json'
selection_path.write_text(json.dumps(selection_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)) + '\n', encoding='utf-8')

readout = '\n'.join(readout_lines)
readout_path = CARD_DIR / 'readout.md'
readout_path.write_text(readout + '\n', encoding='utf-8')

display(Markdown(readout))
print('wrote', selection_path.relative_to(ROOT), readout_path.relative_to(ROOT))


## Final Readout

This notebook is the primary single-run conclusion artifact.

- Decision: `diagnostic_next_step_rebound_router_no_promotion`
- Next mechanism: `state-conditioned rebound participation router with separate high-vol positive-rebound and high-vol downtrend branches`
- Repeated poor-window clusters: `2026_high_vol_rebound_underexposure, 2025_high_vol_rebound_underexposure`
- Top feature group: `volatility_path`
- Cap-like sources still fail rebound excess: `True`
- Strict MaxDD pass sources among structured selections: `0`

### Mechanism Notes

- The persistent 2025/2026 clusters have large negative excess while drawdown is often already below the 15% target, so the remaining failure is underparticipation in violent rebounds, not only account drawdown.
- Feature consensus is dominated by volatility-path and trend/rebound variables. `vol_of_vol_21`, `jump_variation_5`, `realized_vol_63`, and trend/rebound terms recur across cards.
- Tightening execution caps alone is now a duplicated failure mode: it leaves rebound underexposure while risking `wf_2022` or trend-window regression.
- The next experiment should route state, not tune another scalar cap: allow rebound participation only when high-volatility positive momentum and long-anchor recovery are both present; de-risk separately when high volatility is paired with trend breaks/downside momentum.

### Artifacts

- `runtime/execution/card/poor_window_mechanism_router_v1/poor_window_inventory.csv`
- `runtime/execution/card/poor_window_mechanism_router_v1/cluster_summary.csv`
- `runtime/execution/card/poor_window_mechanism_router_v1/feature_consensus.csv`
- `runtime/execution/card/poor_window_mechanism_router_v1/feature_group_consensus.csv`
- `runtime/execution/card/poor_window_mechanism_router_v1/rebound_cluster_comparison.csv`

wrote runtime/execution/card/poor_window_mechanism_router_v1/selection.json runtime/execution/card/poor_window_mechanism_router_v1/readout.md
